# React E-Commerce Store with Stripe Payment Integration

This notebook teaches you how to build a complete e-commerce store with real payment processing using Stripe.

## What You'll Learn:
- Product catalog management
- Shopping cart functionality
- Stripe payment integration
- Checkout process
- Order confirmation
- Payment webhooks

## Prerequisites:
- Node.js installed
- Stripe account (get free test keys at https://stripe.com)
- Basic React knowledge

Run cells one by one from top to bottom.


## 1. Project Setup and Dependencies

First, let's install the required packages for our store.


In [ ]:
// Run this in your terminal:
// npm install react react-dom
// npm install @stripe/stripe-js @stripe/react-stripe-js
// npm install axios
// npm install express stripe cors dotenv (for backend)

console.log('Dependencies to install:');
console.log('Frontend: @stripe/stripe-js, @stripe/react-stripe-js, axios');
console.log('Backend: express, stripe, cors, dotenv');


## 2. Environment Configuration

Create a `.env` file in your project root with your Stripe keys.


In [ ]:
// .env file content:
/*
REACT_APP_STRIPE_PUBLISHABLE_KEY=pk_test_your_publishable_key_here
STRIPE_SECRET_KEY=sk_test_your_secret_key_here
*/

// Access in React:
const stripePublishableKey = process.env.REACT_APP_STRIPE_PUBLISHABLE_KEY;

console.log('Get your Stripe keys from: https://dashboard.stripe.com/test/apikeys');
console.log('Use TEST keys (pk_test_... and sk_test_...) for development');


## 3. Product Data Structure

Define your product catalog with prices in cents (Stripe uses smallest currency unit).


In [ ]:
// products.js
export const products = [
  {
    id: 'prod_1',
    name: 'Premium Headphones',
    description: 'High-quality wireless headphones with noise cancellation',
    price: 29999, // $299.99 in cents
    currency: 'usd',
    image: 'https://via.placeholder.com/300x300?text=Headphones',
    stripePriceId: 'price_xxxxxxxxxxxxx' // Create in Stripe Dashboard
  },
  {
    id: 'prod_2',
    name: 'Smart Watch',
    description: 'Fitness tracker with heart rate monitor',
    price: 19999, // $199.99
    currency: 'usd',
    image: 'https://via.placeholder.com/300x300?text=Smart+Watch',
    stripePriceId: 'price_xxxxxxxxxxxxx'
  },
  {
    id: 'prod_3',
    name: 'Laptop Stand',
    description: 'Ergonomic aluminum laptop stand',
    price: 4999, // $49.99
    currency: 'usd',
    image: 'https://via.placeholder.com/300x300?text=Laptop+Stand',
    stripePriceId: 'price_xxxxxxxxxxxxx'
  },
  {
    id: 'prod_4',
    name: 'Mechanical Keyboard',
    description: 'RGB backlit mechanical gaming keyboard',
    price: 12999, // $129.99
    currency: 'usd',
    image: 'https://via.placeholder.com/300x300?text=Keyboard',
    stripePriceId: 'price_xxxxxxxxxxxxx'
  }
];

// Helper function to format price
export const formatPrice = (cents, currency = 'usd') => {
  return new Intl.NumberFormat('en-US', {
    style: 'currency',
    currency: currency.toUpperCase()
  }).format(cents / 100);
};


## 4. Shopping Cart Context

Create a context to manage cart state across the application.


In [ ]:
// CartContext.js
import React, { createContext, useContext, useReducer } from 'react';

const CartContext = createContext();

const cartReducer = (state, action) => {
  switch (action.type) {
    case 'ADD_TO_CART':
      const existingItem = state.items.find(item => item.id === action.payload.id);
      
      if (existingItem) {
        return {
          ...state,
          items: state.items.map(item =>
            item.id === action.payload.id
              ? { ...item, quantity: item.quantity + 1 }
              : item
          )
        };
      }
      
      return {
        ...state,
        items: [...state.items, { ...action.payload, quantity: 1 }]
      };
    
    case 'REMOVE_FROM_CART':
      return {
        ...state,
        items: state.items.filter(item => item.id !== action.payload)
      };
    
    case 'UPDATE_QUANTITY':
      return {
        ...state,
        items: state.items.map(item =>
          item.id === action.payload.id
            ? { ...item, quantity: action.payload.quantity }
            : item
        )
      };
    
    case 'CLEAR_CART':
      return { ...state, items: [] };
    
    default:
      return state;
  }
};

export const CartProvider = ({ children }) => {
  const [state, dispatch] = useReducer(cartReducer, { items: [] });
  
  const addToCart = (product) => {
    dispatch({ type: 'ADD_TO_CART', payload: product });
  };
  
  const removeFromCart = (productId) => {
    dispatch({ type: 'REMOVE_FROM_CART', payload: productId });
  };
  
  const updateQuantity = (productId, quantity) => {
    if (quantity <= 0) {
      removeFromCart(productId);
    } else {
      dispatch({ type: 'UPDATE_QUANTITY', payload: { id: productId, quantity } });
    }
  };
  
  const clearCart = () => {
    dispatch({ type: 'CLEAR_CART' });
  };
  
  const getCartTotal = () => {
    return state.items.reduce((total, item) => total + (item.price * item.quantity), 0);
  };
  
  const getCartCount = () => {
    return state.items.reduce((count, item) => count + item.quantity, 0);
  };
  
  return (
    <CartContext.Provider value={{
      items: state.items,
      addToCart,
      removeFromCart,
      updateQuantity,
      clearCart,
      getCartTotal,
      getCartCount
    }}>
      {children}
    </CartContext.Provider>
  );
};

export const useCart = () => {
  const context = useContext(CartContext);
  if (!context) {
    throw new Error('useCart must be used within CartProvider');
  }
  return context;
};


## 5. Product Card Component

Display individual products with add to cart functionality.


In [ ]:
// ProductCard.js
import React from 'react';
import { useCart } from './CartContext';
import { formatPrice } from './products';

const ProductCard = ({ product }) => {
  const { addToCart } = useCart();
  
  const handleAddToCart = () => {
    addToCart(product);
    alert(`${product.name} added to cart!`);
  };
  
  return (
    <div style={{
      border: '1px solid #ddd',
      borderRadius: '8px',
      padding: '20px',
      textAlign: 'center',
      transition: 'transform 0.2s',
      cursor: 'pointer'
    }}
    onMouseEnter={(e) => e.currentTarget.style.transform = 'scale(1.05)'}
    onMouseLeave={(e) => e.currentTarget.style.transform = 'scale(1)'}>
      <img 
        src={product.image} 
        alt={product.name}
        style={{ width: '100%', height: '200px', objectFit: 'cover', borderRadius: '4px' }}
      />
      <h3 style={{ margin: '15px 0 10px' }}>{product.name}</h3>
      <p style={{ color: '#666', fontSize: '14px', minHeight: '40px' }}>
        {product.description}
      </p>
      <p style={{ fontSize: '24px', fontWeight: 'bold', color: '#2c3e50', margin: '10px 0' }}>
        {formatPrice(product.price, product.currency)}
      </p>
      <button
        onClick={handleAddToCart}
        style={{
          backgroundColor: '#4CAF50',
          color: 'white',
          border: 'none',
          padding: '12px 24px',
          borderRadius: '4px',
          cursor: 'pointer',
          fontSize: '16px',
          width: '100%',
          transition: 'background-color 0.2s'
        }}
        onMouseEnter={(e) => e.target.style.backgroundColor = '#45a049'}
        onMouseLeave={(e) => e.target.style.backgroundColor = '#4CAF50'}>
        Add to Cart
      </button>
    </div>
  );
};

export default ProductCard;


## 6. Product List Component

Display all products in a grid layout.


In [ ]:
// ProductList.js
import React from 'react';
import ProductCard from './ProductCard';
import { products } from './products';

const ProductList = () => {
  return (
    <div style={{ padding: '20px' }}>
      <h1 style={{ textAlign: 'center', marginBottom: '30px' }}>Our Products</h1>
      <div style={{
        display: 'grid',
        gridTemplateColumns: 'repeat(auto-fill, minmax(280px, 1fr))',
        gap: '20px',
        maxWidth: '1200px',
        margin: '0 auto'
      }}>
        {products.map(product => (
          <ProductCard key={product.id} product={product} />
        ))}
      </div>
    </div>
  );
};

export default ProductList;


## 7. Shopping Cart Component

Display cart items with quantity controls and total.


In [ ]:
// ShoppingCart.js
import React from 'react';
import { useCart } from './CartContext';
import { formatPrice } from './products';

const ShoppingCart = ({ onCheckout }) => {
  const { items, removeFromCart, updateQuantity, getCartTotal, clearCart } = useCart();
  
  if (items.length === 0) {
    return (
      <div style={{ padding: '40px', textAlign: 'center' }}>
        <h2>Your cart is empty</h2>
        <p>Add some products to get started!</p>
      </div>
    );
  }
  
  return (
    <div style={{ padding: '20px', maxWidth: '800px', margin: '0 auto' }}>
      <h2>Shopping Cart</h2>
      
      {items.map(item => (
        <div key={item.id} style={{
          display: 'flex',
          alignItems: 'center',
          padding: '15px',
          borderBottom: '1px solid #ddd',
          gap: '15px'
        }}>
          <img 
            src={item.image} 
            alt={item.name}
            style={{ width: '80px', height: '80px', objectFit: 'cover', borderRadius: '4px' }}
          />
          
          <div style={{ flex: 1 }}>
            <h3 style={{ margin: '0 0 5px' }}>{item.name}</h3>
            <p style={{ margin: 0, color: '#666' }}>
              {formatPrice(item.price, item.currency)}
            </p>
          </div>
          
          <div style={{ display: 'flex', alignItems: 'center', gap: '10px' }}>
            <button
              onClick={() => updateQuantity(item.id, item.quantity - 1)}
              style={{ padding: '5px 10px', cursor: 'pointer' }}>
              -
            </button>
            <span style={{ minWidth: '30px', textAlign: 'center' }}>{item.quantity}</span>
            <button
              onClick={() => updateQuantity(item.id, item.quantity + 1)}
              style={{ padding: '5px 10px', cursor: 'pointer' }}>
              +
            </button>
          </div>
          
          <p style={{ fontWeight: 'bold', minWidth: '80px', textAlign: 'right' }}>
            {formatPrice(item.price * item.quantity, item.currency)}
          </p>
          
          <button
            onClick={() => removeFromCart(item.id)}
            style={{
              backgroundColor: '#f44336',
              color: 'white',
              border: 'none',
              padding: '8px 12px',
              borderRadius: '4px',
              cursor: 'pointer'
            }}>
            Remove
          </button>
        </div>
      ))}
      
      <div style={{ marginTop: '20px', textAlign: 'right' }}>
        <h3>Total: {formatPrice(getCartTotal())}</h3>
        <div style={{ display: 'flex', gap: '10px', justifyContent: 'flex-end', marginTop: '15px' }}>
          <button
            onClick={clearCart}
            style={{
              padding: '12px 24px',
              backgroundColor: '#999',
              color: 'white',
              border: 'none',
              borderRadius: '4px',
              cursor: 'pointer'
            }}>
            Clear Cart
          </button>
          <button
            onClick={onCheckout}
            style={{
              padding: '12px 24px',
              backgroundColor: '#4CAF50',
              color: 'white',
              border: 'none',
              borderRadius: '4px',
              cursor: 'pointer',
              fontSize: '16px'
            }}>
            Proceed to Checkout
          </button>
        </div>
      </div>
    </div>
  );
};

export default ShoppingCart;


## 8. Backend Server - Express with Stripe

Create a Node.js/Express server to handle Stripe payments securely.


In [ ]:
// server.js
const express = require('express');
const cors = require('cors');
const stripe = require('stripe')(process.env.STRIPE_SECRET_KEY);
require('dotenv').config();

const app = express();

app.use(cors());
app.use(express.json());

// Create payment intent
app.post('/create-payment-intent', async (req, res) => {
  try {
    const { amount, currency = 'usd', metadata } = req.body;
    
    // Create a PaymentIntent with the order amount and currency
    const paymentIntent = await stripe.paymentIntents.create({
      amount,
      currency,
      metadata,
      automatic_payment_methods: {
        enabled: true,
      },
    });
    
    res.json({
      clientSecret: paymentIntent.client_secret,
      paymentIntentId: paymentIntent.id
    });
  } catch (error) {
    res.status(400).json({ error: error.message });
  }
});

// Create checkout session (alternative method)
app.post('/create-checkout-session', async (req, res) => {
  try {
    const { items } = req.body;
    
    const session = await stripe.checkout.sessions.create({
      payment_method_types: ['card'],
      line_items: items.map(item => ({
        price_data: {
          currency: 'usd',
          product_data: {
            name: item.name,
            description: item.description,
            images: [item.image],
          },
          unit_amount: item.price,
        },
        quantity: item.quantity,
      })),
      mode: 'payment',
      success_url: `${process.env.CLIENT_URL}/success?session_id={CHECKOUT_SESSION_ID}`,
      cancel_url: `${process.env.CLIENT_URL}/cancel`,
    });
    
    res.json({ sessionId: session.id, url: session.url });
  } catch (error) {
    res.status(400).json({ error: error.message });
  }
});

// Webhook to handle Stripe events
app.post('/webhook', express.raw({ type: 'application/json' }), async (req, res) => {
  const sig = req.headers['stripe-signature'];
  const webhookSecret = process.env.STRIPE_WEBHOOK_SECRET;
  
  let event;
  
  try {
    event = stripe.webhooks.constructEvent(req.body, sig, webhookSecret);
  } catch (err) {
    console.log(`Webhook Error: ${err.message}`);
    return res.status(400).send(`Webhook Error: ${err.message}`);
  }
  
  // Handle the event
  switch (event.type) {
    case 'payment_intent.succeeded':
      const paymentIntent = event.data.object;
      console.log('PaymentIntent was successful!', paymentIntent.id);
      // Fulfill the order, send confirmation email, etc.
      break;
    
    case 'payment_intent.payment_failed':
      const failedPayment = event.data.object;
      console.log('Payment failed:', failedPayment.id);
      // Notify customer of failed payment
      break;
    
    case 'checkout.session.completed':
      const session = event.data.object;
      console.log('Checkout session completed:', session.id);
      // Fulfill the order
      break;
    
    default:
      console.log(`Unhandled event type ${event.type}`);
  }
  
  res.json({ received: true });
});

// Get session details
app.get('/session/:sessionId', async (req, res) => {
  try {
    const session = await stripe.checkout.sessions.retrieve(req.params.sessionId);
    res.json(session);
  } catch (error) {
    res.status(400).json({ error: error.message });
  }
});

const PORT = process.env.PORT || 3001;
app.listen(PORT, () => {
  console.log(`Server running on port ${PORT}`);
});


## 9. Stripe Elements - Payment Form Component

Create a secure payment form using Stripe Elements.


In [ ]:
// CheckoutForm.js
import React, { useState } from 'react';
import { CardElement, useStripe, useElements } from '@stripe/react-stripe-js';
import axios from 'axios';
import { useCart } from './CartContext';
import { formatPrice } from './products';

const CARD_ELEMENT_OPTIONS = {
  style: {
    base: {
      color: '#32325d',
      fontFamily: '"Helvetica Neue", Helvetica, sans-serif',
      fontSmoothing: 'antialiased',
      fontSize: '16px',
      '::placeholder': {
        color: '#aab7c4'
      }
    },
    invalid: {
      color: '#fa755a',
      iconColor: '#fa755a'
    }
  }
};

const CheckoutForm = ({ onSuccess }) => {
  const stripe = useStripe();
  const elements = useElements();
  const { items, getCartTotal, clearCart } = useCart();
  
  const [processing, setProcessing] = useState(false);
  const [error, setError] = useState(null);
  const [billingDetails, setBillingDetails] = useState({
    name: '',
    email: '',
    address: {
      line1: '',
      city: '',
      state: '',
      postal_code: '',
      country: 'US'
    }
  });
  
  const handleInputChange = (e) => {
    const { name, value } = e.target;
    
    if (name.startsWith('address.')) {
      const addressField = name.split('.')[1];
      setBillingDetails(prev => ({
        ...prev,
        address: { ...prev.address, [addressField]: value }
      }));
    } else {
      setBillingDetails(prev => ({ ...prev, [name]: value }));
    }
  };
  
  const handleSubmit = async (e) => {
    e.preventDefault();
    
    if (!stripe || !elements) {
      return;
    }
    
    setProcessing(true);
    setError(null);
    
    try {
      // Create payment intent on backend
      const { data } = await axios.post('http://localhost:3001/create-payment-intent', {
        amount: getCartTotal(),
        currency: 'usd',
        metadata: {
          items: JSON.stringify(items.map(item => ({
            id: item.id,
            name: item.name,
            quantity: item.quantity
          })))
        }
      });
      
      // Confirm payment with Stripe
      const { error: stripeError, paymentIntent } = await stripe.confirmCardPayment(
        data.clientSecret,
        {
          payment_method: {
            card: elements.getElement(CardElement),
            billing_details: billingDetails
          }
        }
      );
      
      if (stripeError) {
        setError(stripeError.message);
        setProcessing(false);
        return;
      }
      
      if (paymentIntent.status === 'succeeded') {
        clearCart();
        onSuccess(paymentIntent);
      }
    } catch (err) {
      setError(err.message);
      setProcessing(false);
    }
  };
  
  return (
    <form onSubmit={handleSubmit} style={{ maxWidth: '500px', margin: '0 auto', padding: '20px' }}>
      <h2>Checkout</h2>
      
      <div style={{ marginBottom: '20px' }}>
        <h3>Order Summary</h3>
        {items.map(item => (
          <div key={item.id} style={{ display: 'flex', justifyContent: 'space-between', marginBottom: '10px' }}>
            <span>{item.name} x {item.quantity}</span>
            <span>{formatPrice(item.price * item.quantity)}</span>
          </div>
        ))}
        <hr />
        <div style={{ display: 'flex', justifyContent: 'space-between', fontWeight: 'bold', fontSize: '18px' }}>
          <span>Total:</span>
          <span>{formatPrice(getCartTotal())}</span>
        </div>
      </div>
      
      <div style={{ marginBottom: '15px' }}>
        <label>Name</label>
        <input
          type="text"
          name="name"
          value={billingDetails.name}
          onChange={handleInputChange}
          required
          style={{ width: '100%', padding: '10px', marginTop: '5px' }}
        />
      </div>
      
      <div style={{ marginBottom: '15px' }}>
        <label>Email</label>
        <input
          type="email"
          name="email"
          value={billingDetails.email}
          onChange={handleInputChange}
          required
          style={{ width: '100%', padding: '10px', marginTop: '5px' }}
        />
      </div>
      
      <div style={{ marginBottom: '15px' }}>
        <label>Card Details</label>
        <div style={{ padding: '12px', border: '1px solid #ccc', borderRadius: '4px', marginTop: '5px' }}>
          <CardElement options={CARD_ELEMENT_OPTIONS} />
        </div>
      </div>
      
      {error && (
        <div style={{ color: 'red', marginBottom: '15px', padding: '10px', backgroundColor: '#ffebee', borderRadius: '4px' }}>
          {error}
        </div>
      )}
      
      <button
        type="submit"
        disabled={!stripe || processing}
        style={{
          width: '100%',
          padding: '15px',
          backgroundColor: processing ? '#ccc' : '#4CAF50',
          color: 'white',
          border: 'none',
          borderRadius: '4px',
          fontSize: '16px',
          cursor: processing ? 'not-allowed' : 'pointer'
        }}>
        {processing ? 'Processing...' : `Pay ${formatPrice(getCartTotal())}`}
      </button>
      
      <p style={{ marginTop: '15px', fontSize: '12px', color: '#666', textAlign: 'center' }}>
        Test card: 4242 4242 4242 4242 | Any future date | Any 3 digits
      </p>
    </form>
  );
};

export default CheckoutForm;


## 10. Stripe Provider Setup

Wrap your app with Stripe's Elements provider.


In [ ]:
// StripeWrapper.js
import React from 'react';
import { Elements } from '@stripe/react-stripe-js';
import { loadStripe } from '@stripe/stripe-js';
import CheckoutForm from './CheckoutForm';

// Load Stripe outside component to avoid recreating on every render
const stripePromise = loadStripe(process.env.REACT_APP_STRIPE_PUBLISHABLE_KEY);

const StripeWrapper = ({ onSuccess }) => {
  const options = {
    // Customize appearance
    appearance: {
      theme: 'stripe',
      variables: {
        colorPrimary: '#4CAF50',
      },
    },
  };
  
  return (
    <Elements stripe={stripePromise} options={options}>
      <CheckoutForm onSuccess={onSuccess} />
    </Elements>
  );
};

export default StripeWrapper;


## 11. Success Page Component

Display order confirmation after successful payment.


In [ ]:
// SuccessPage.js
import React from 'react';
import { formatPrice } from './products';

const SuccessPage = ({ paymentIntent }) => {
  return (
    <div style={{
      maxWidth: '600px',
      margin: '50px auto',
      padding: '40px',
      textAlign: 'center',
      backgroundColor: '#f0f9ff',
      borderRadius: '8px',
      border: '2px solid #4CAF50'
    }}>
      <div style={{ fontSize: '64px', marginBottom: '20px' }}>✓</div>
      <h1 style={{ color: '#4CAF50', marginBottom: '10px' }}>Payment Successful!</h1>
      <p style={{ fontSize: '18px', color: '#666', marginBottom: '30px' }}>
        Thank you for your purchase!
      </p>
      
      {paymentIntent && (
        <div style={{
          backgroundColor: 'white',
          padding: '20px',
          borderRadius: '4px',
          marginBottom: '20px',
          textAlign: 'left'
        }}>
          <h3>Order Details</h3>
          <p><strong>Payment ID:</strong> {paymentIntent.id}</p>
          <p><strong>Amount:</strong> {formatPrice(paymentIntent.amount)}</p>
          <p><strong>Status:</strong> <span style={{ color: '#4CAF50' }}>Paid</span></p>
        </div>
      )}
      
      <p style={{ marginBottom: '20px' }}>
        A confirmation email has been sent to your email address.
      </p>
      
      <button
        onClick={() => window.location.href = '/'}
        style={{
          padding: '12px 24px',
          backgroundColor: '#4CAF50',
          color: 'white',
          border: 'none',
          borderRadius: '4px',
          fontSize: '16px',
          cursor: 'pointer'
        }}>
        Continue Shopping
      </button>
    </div>
  );
};

export default SuccessPage;


## 12. Main App Component

Bring everything together in the main App component.


In [ ]:
// App.js
import React, { useState } from 'react';
import { CartProvider, useCart } from './CartContext';
import ProductList from './ProductList';
import ShoppingCart from './ShoppingCart';
import StripeWrapper from './StripeWrapper';
import SuccessPage from './SuccessPage';

const Header = ({ onCartClick, onHomeClick }) => {
  const { getCartCount } = useCart();
  
  return (
    <header style={{
      backgroundColor: '#2c3e50',
      color: 'white',
      padding: '20px',
      display: 'flex',
      justifyContent: 'space-between',
      alignItems: 'center'
    }}>
      <h1 
        onClick={onHomeClick}
        style={{ margin: 0, cursor: 'pointer' }}>
        🛒 React Stripe Store
      </h1>
      <button
        onClick={onCartClick}
        style={{
          backgroundColor: '#4CAF50',
          color: 'white',
          border: 'none',
          padding: '10px 20px',
          borderRadius: '4px',
          cursor: 'pointer',
          fontSize: '16px',
          position: 'relative'
        }}>
        Cart
        {getCartCount() > 0 && (
          <span style={{
            position: 'absolute',
            top: '-8px',
            right: '-8px',
            backgroundColor: 'red',
            color: 'white',
            borderRadius: '50%',
            width: '24px',
            height: '24px',
            display: 'flex',
            alignItems: 'center',
            justifyContent: 'center',
            fontSize: '12px'
          }}>
            {getCartCount()}
          </span>
        )}
      </button>
    </header>
  );
};

const AppContent = () => {
  const [currentView, setCurrentView] = useState('products'); // products, cart, checkout, success
  const [paymentIntent, setPaymentIntent] = useState(null);
  
  const handleCheckout = () => {
    setCurrentView('checkout');
  };
  
  const handlePaymentSuccess = (intent) => {
    setPaymentIntent(intent);
    setCurrentView('success');
  };
  
  return (
    <div>
      <Header 
        onCartClick={() => setCurrentView('cart')}
        onHomeClick={() => setCurrentView('products')}
      />
      
      {currentView === 'products' && <ProductList />}
      {currentView === 'cart' && <ShoppingCart onCheckout={handleCheckout} />}
      {currentView === 'checkout' && <StripeWrapper onSuccess={handlePaymentSuccess} />}
      {currentView === 'success' && <SuccessPage paymentIntent={paymentIntent} />}
    </div>
  );
};

function App() {
  return (
    <CartProvider>
      <AppContent />
    </CartProvider>
  );
}

export default App;


## 13. Entry Point - index.js

Set up the React app entry point.


In [ ]:
// index.js
import React from 'react';
import ReactDOM from 'react-dom/client';
import App from './App';

const root = ReactDOM.createRoot(document.getElementById('root'));
root.render(
  <React.StrictMode>
    <App />
  </React.StrictMode>
);


## 14. Package.json Configuration

Complete package.json with all dependencies.


In [ ]:
// package.json
{
  "name": "react-stripe-store",
  "version": "1.0.0",
  "description": "E-commerce store with Stripe integration",
  "scripts": {
    "start": "react-scripts start",
    "build": "react-scripts build",
    "server": "node server.js",
    "dev": "concurrently \"npm start\" \"npm run server\""
  },
  "dependencies": {
    "react": "^18.2.0",
    "react-dom": "^18.2.0",
    "@stripe/stripe-js": "^2.2.0",
    "@stripe/react-stripe-js": "^2.4.0",
    "axios": "^1.6.0",
    "express": "^4.18.2",
    "stripe": "^14.5.0",
    "cors": "^2.8.5",
    "dotenv": "^16.3.1"
  },
  "devDependencies": {
    "react-scripts": "^5.0.1",
    "concurrently": "^8.2.2"
  }
}


## 15. Setup Instructions

Step-by-step guide to run the store.


In [ ]:
/*
SETUP INSTRUCTIONS:

1. Create Stripe Account:
   - Go to https://stripe.com
   - Sign up for a free account
   - Get your test API keys from Dashboard > Developers > API keys

2. Create Project:
   npx create-react-app react-stripe-store
   cd react-stripe-store

3. Install Dependencies:
   npm install @stripe/stripe-js @stripe/react-stripe-js axios
   npm install express stripe cors dotenv
   npm install --save-dev concurrently

4. Create .env file in root:
   REACT_APP_STRIPE_PUBLISHABLE_KEY=pk_test_your_key_here
   STRIPE_SECRET_KEY=sk_test_your_key_here
   CLIENT_URL=http://localhost:3000
   PORT=3001

5. Create all component files as shown in previous cells

6. Run the application:
   # Terminal 1 - Frontend
   npm start
   
   # Terminal 2 - Backend
   npm run server
   
   # Or run both together:
   npm run dev

7. Test with Stripe test cards:
   - Success: 4242 4242 4242 4242
   - Decline: 4000 0000 0000 0002
   - 3D Secure: 4000 0025 0000 3155
   - Use any future expiry date and any 3-digit CVC

8. Set up webhooks (for production):
   - Install Stripe CLI: https://stripe.com/docs/stripe-cli
   - Run: stripe listen --forward-to localhost:3001/webhook
   - Copy webhook signing secret to .env as STRIPE_WEBHOOK_SECRET
*/

console.log('Follow the setup instructions above to run the store!');


## 16. Testing Your Store

How to test payments without real money.


In [ ]:
/*
STRIPE TEST CARDS:

1. Successful Payment:
   Card: 4242 4242 4242 4242
   Expiry: Any future date (e.g., 12/25)
   CVC: Any 3 digits (e.g., 123)
   ZIP: Any 5 digits (e.g., 12345)

2. Payment Declined:
   Card: 4000 0000 0000 0002
   Result: Generic decline

3. Insufficient Funds:
   Card: 4000 0000 0000 9995
   Result: Card declined due to insufficient funds

4. 3D Secure Authentication:
   Card: 4000 0025 0000 3155
   Result: Requires authentication

5. Expired Card:
   Card: 4000 0000 0000 0069
   Result: Card expired

TESTING CHECKLIST:
□ Add products to cart
□ Update quantities
□ Remove items
□ Clear cart
□ Proceed to checkout
□ Fill billing details
□ Test successful payment
□ Test declined payment
□ Verify order confirmation
□ Check webhook events in Stripe Dashboard
□ Test with different currencies (if applicable)
□ Test mobile responsiveness
*/

console.log('Use Stripe test cards to test payments safely!');


## 17. Security Best Practices

Important security considerations for production.


In [ ]:
/*
SECURITY BEST PRACTICES:

1. API Keys:
   ✓ NEVER commit API keys to version control
   ✓ Use environment variables (.env)
   ✓ Add .env to .gitignore
   ✓ Use different keys for test and production
   ✓ Rotate keys periodically

2. Backend Security:
   ✓ Always create payment intents on the server
   ✓ Never expose secret keys to frontend
   ✓ Validate amounts on the server
   ✓ Implement rate limiting
   ✓ Use HTTPS in production
   ✓ Validate webhook signatures

3. Frontend Security:
   ✓ Use Stripe Elements (PCI compliant)
   ✓ Never handle raw card data
   ✓ Validate user input
   ✓ Implement CSRF protection
   ✓ Use Content Security Policy headers

4. Data Protection:
   ✓ Don't store card details
   ✓ Use Stripe's tokenization
   ✓ Encrypt sensitive data at rest
   ✓ Use secure database connections
   ✓ Implement proper access controls

5. Webhook Security:
   ✓ Verify webhook signatures
   ✓ Use HTTPS endpoints
   ✓ Implement idempotency
   ✓ Handle duplicate events
   ✓ Log all webhook events

6. Error Handling:
   ✓ Don't expose sensitive errors to users
   ✓ Log errors securely
   ✓ Implement proper error boundaries
   ✓ Handle network failures gracefully
*/

// Example: Webhook signature verification
const verifyWebhookSignature = (payload, signature, secret) => {
  try {
    const event = stripe.webhooks.constructEvent(payload, signature, secret);
    return event;
  } catch (err) {
    console.error('Webhook signature verification failed:', err.message);
    throw new Error('Invalid signature');
  }
};

// Example: Server-side amount validation
const validatePaymentAmount = (items) => {
  const calculatedTotal = items.reduce((sum, item) => {
    const product = products.find(p => p.id === item.id);
    if (!product) throw new Error('Invalid product');
    return sum + (product.price * item.quantity);
  }, 0);
  return calculatedTotal;
};


## 18. Deployment Guide

Deploy your store to production.


In [ ]:
/*
DEPLOYMENT STEPS:

1. FRONTEND (Vercel/Netlify):
   
   Vercel:
   - Install Vercel CLI: npm i -g vercel
   - Run: vercel
   - Add environment variables in Vercel dashboard
   - Set REACT_APP_STRIPE_PUBLISHABLE_KEY
   
   Netlify:
   - Install Netlify CLI: npm i -g netlify-cli
   - Run: netlify deploy
   - Add environment variables in Netlify dashboard

2. BACKEND (Heroku/Railway/Render):
   
   Heroku:
   - Create Heroku app: heroku create your-app-name
   - Set environment variables:
     heroku config:set STRIPE_SECRET_KEY=sk_live_...
     heroku config:set STRIPE_WEBHOOK_SECRET=whsec_...
   - Deploy: git push heroku main
   
   Railway:
   - Connect GitHub repo
   - Add environment variables in dashboard
   - Deploy automatically on push

3. STRIPE PRODUCTION SETUP:
   - Switch to live mode in Stripe Dashboard
   - Get live API keys (pk_live_... and sk_live_...)
   - Update environment variables
   - Set up production webhooks:
     * Go to Developers > Webhooks
     * Add endpoint: https://your-api.com/webhook
     * Select events: payment_intent.succeeded, etc.
     * Copy webhook signing secret

4. DNS & DOMAIN:
   - Purchase domain (Namecheap, Google Domains)
   - Configure DNS records
   - Set up SSL certificate (automatic with Vercel/Netlify)

5. MONITORING:
   - Set up error tracking (Sentry)
   - Monitor Stripe Dashboard for payments
   - Set up uptime monitoring (UptimeRobot)
   - Configure email notifications

6. PRE-LAUNCH CHECKLIST:
   □ Test all payment flows in production mode
   □ Verify webhook endpoints are working
   □ Check SSL certificate is valid
   □ Test on multiple devices/browsers
   □ Review Stripe compliance requirements
   □ Set up customer support email
   □ Create privacy policy and terms of service
   □ Test refund process
   □ Verify email notifications work
   □ Check analytics are tracking correctly
*/

console.log('Ready to deploy your store!');


## 19. Advanced Features to Add

Enhance your store with these features.


In [ ]:
/*
ADVANCED FEATURES:

1. User Authentication:
   - Firebase Auth / Auth0
   - User profiles
   - Order history
   - Saved payment methods

2. Product Management:
   - Admin dashboard
   - Inventory tracking
   - Product variants (size, color)
   - Product reviews
   - Search and filters

3. Payment Features:
   - Multiple currencies
   - Subscription billing
   - Discount codes/coupons
   - Gift cards
   - Split payments
   - Apple Pay / Google Pay

4. Shipping:
   - Shipping calculator
   - Multiple shipping options
   - Address validation
   - Tracking integration
   - International shipping

5. Email Notifications:
   - Order confirmation
   - Shipping updates
   - Abandoned cart recovery
   - Receipt emails
   - Marketing emails

6. Analytics:
   - Google Analytics
   - Conversion tracking
   - Revenue reports
   - Customer insights
   - A/B testing

7. Customer Support:
   - Live chat (Intercom, Zendesk)
   - FAQ section
   - Refund management
   - Dispute handling

8. Performance:
   - Image optimization
   - Lazy loading
   - CDN integration
   - Caching strategies
   - Code splitting

9. SEO:
   - Meta tags
   - Structured data
   - Sitemap
   - Open Graph tags
   - Product schema markup

10. Mobile App:
    - React Native version
    - Push notifications
    - Mobile wallet integration
*/

console.log('Lots of features to explore!');


## 20. Resources and Next Steps

Continue learning and building.


In [ ]:
/*
RESOURCES:

Official Documentation:
- Stripe Docs: https://stripe.com/docs
- Stripe React: https://stripe.com/docs/stripe-js/react
- Stripe API Reference: https://stripe.com/docs/api
- React Docs: https://react.dev

Stripe Tools:
- Stripe Dashboard: https://dashboard.stripe.com
- Stripe CLI: https://stripe.com/docs/stripe-cli
- Stripe Testing: https://stripe.com/docs/testing
- Webhook Testing: https://stripe.com/docs/webhooks/test

Learning Resources:
- Stripe YouTube Channel
- Stripe Blog
- React + Stripe tutorials
- E-commerce best practices

Community:
- Stripe Discord
- Stack Overflow (stripe tag)
- Reddit r/stripe
- GitHub discussions

NEXT STEPS:
1. Build your product catalog
2. Customize the design
3. Add your branding
4. Test thoroughly
5. Deploy to production
6. Market your store
7. Iterate based on feedback

CONGRATULATIONS! 🎉
You now have a fully functional e-commerce store with real payment processing!

Remember:
- Start with test mode
- Test all payment scenarios
- Follow security best practices
- Monitor your Stripe Dashboard
- Provide excellent customer support

Happy selling! 💰
*/

console.log('You\'re ready to build your e-commerce empire!');
